In [23]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

In [24]:
def get_properties(uri):

    query="""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX : <http://publications.europa.eu/ontology/cdm#>

    SELECT ?property ?label ?comment
    WHERE 
    {{ ?property rdfs:domain <"""+uri+""">;
         rdfs:label ?label;
         rdfs:comment ?comment}
    }
    """

    return exec_query(query)

In [25]:
def get_subclasses(uri):

    query="""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl:  <http://www.w3.org/2002/07/owl#>
    
    SELECT DISTINCT ?subclass ?label ?comment
    WHERE 
    { ?subclass rdfs:subClassOf <"""+uri+""">;
        rdfs:label ?label;
        rdfs:comment ?comment
        }
    """

    return exec_query(query)

In [26]:
def exec_query(query):
    sparql = SPARQLWrapper("https://publications.europa.eu/webapi/rdf/sparql")

    sparql.setQuery(query)

    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    return results

## OLD

In [ ]:
def get_results(uri,typeQuery,list_subclasses):
    df_total=pd.DataFrame()
    results=get_subclasses(uri)
    if(len(results["results"]["bindings"])!=0):
        for r in results["results"]["bindings"]:
            list_subclasses.append({"class":uri,"type":typeQuery,"element":r[typeQuery]["value"]})
            get_results(r[typeQuery]["value"],"subclass",list_subclasses)
    return list_subclasses

## NEW

In [27]:
def get_results(uri,typeQuery,list_subclasses):
    df_total=pd.DataFrame()
    results=get_subclasses(uri)
    if(len(results["results"]["bindings"])!=0):
        for r in results["results"]["bindings"]:
            list_subclasses.append({"class":uri,"type":typeQuery,"element":r[typeQuery]["value"],"label":r["label"]["value"],"comment":r["comment"]["value"]})
            results_properties=get_properties(r[typeQuery]["value"])
            if(len(results_properties["results"]["bindings"])!=0):
                for p in results_properties["results"]["bindings"]:
                    #print(p)
                    list_subclasses.append({"class":r[typeQuery]["value"],"type":"property","element":p["property"]["value"],"label":p["label"]["value"],"comment":p["comment"]["value"]})
            get_results(r[typeQuery]["value"],"subclass",list_subclasses)
    return list_subclasses

In [28]:
uri='http://publications.europa.eu/ontology/cdm#resource_legal'
list_classes=[]
df=pd.DataFrame(get_results(uri,"subclass",list_classes))
df

,class,type,element,label,comment
0,http://publications.europa.eu/ontology/cdm#res...,subclass,http://publications.europa.eu/ontology/cdm#act...,Consolidated act,Consolidated acts (= sector 0): Non-official d...
1,http://publications.europa.eu/ontology/cdm#act...,property,http://publications.europa.eu/ontology/cdm#act...,Consolidated act based on legal resource,This consolidated act starts out with this leg...
2,http://publications.europa.eu/ontology/cdm#act...,property,http://publications.europa.eu/ontology/cdm#act...,Consolidated act consolidates legal resource,Here according to spec of advanced search the ...
3,http://publications.europa.eu/ontology/cdm#act...,property,http://publications.europa.eu/ontology/cdm#act...,Consolidated act date,Consleg date
4,http://publications.europa.eu/ontology/cdm#act...,property,http://publications.europa.eu/ontology/cdm#act...,Consolidated act layer,Consleg couche. Number of the layer
...,...,...,...,...,...
290,http://publications.europa.eu/ontology/cdm#treaty,subclass,http://publications.europa.eu/ontology/cdm#tre...,Consolidated treaty,Consolidated treaty (= sector 1E in EUR-Lex)
291,http://publications.europa.eu/ontology/cdm#res...,subclass,http://publications.europa.eu/ontology/cdm#tex...,Text adopted,Texts adopted by the European Parliament
292,http://publications.europa.eu/ontology/cdm#tex...,property,http://publications.europa.eu/ontology/cdm#tex...,Parliamentary term,This indicates the Parliamentary term during w...
293,http://publications.europa.eu/ontology/cdm#tex...,subclass,http://publications.europa.eu/ontology/cdm#res...,Legislative resolution,8806 EP legislative resolutions (5AP) \n\nNOT ...


In [9]:
df.iloc[1]["element"]

'http://publications.europa.eu/ontology/cdm#act_consolidated_based_on_resource_legal'

In [ ]:
len(df)

In [ ]:
df["class"].value_counts()

In [ ]:
df["element"].value_counts()

In [ ]:
df

In [29]:
df.to_excel("output.xlsx") 